# Stage 3 — Preprocessing: split, нормализация, Dataset

Что делаем:
1. Train / Val / Test split (70/15/15), стратифицированно
2. Normalization: считаем mean/std **per-feature** на train (развернув последовательности в (N·T, D)) — сохраняем для inference
3. Class weights (sqrt-inverse)
4. `FrameDataset` — отдаёт `(rgb[L,1024], audio[L,128], length, label)` с **dequantize + normalize на лету**. Батч фиксированной длины L, длина используется для масок в RNN.

Важно: данные на диске лежат в `uint8` — экономим в 4 раза. Dequantize делается прямо в `__getitem__`.

In [1]:
# ============================================================
# 3.0 — Импорты и загрузка
# ============================================================
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import pickle, json
from pathlib import Path

SRC_DIR     = Path('../data2/frame_processed')
STAGE3_DIR  = Path('../data2/frame_stage3'); STAGE3_DIR.mkdir(exist_ok=True)

cfg     = json.load(open(SRC_DIR / 'config.json'))
L       = cfg['L']
DIM_RGB = cfg['dim_rgb']
DIM_AUD = cfg['dim_audio']
DEQ_S   = cfg['dequant_scale']
DEQ_B   = cfg['dequant_bias']
GENRES  = cfg['genres']
N_CLASSES = cfg['n_classes']

X_rgb_q   = np.load(SRC_DIR / 'X_rgb.npy',   mmap_mode='r')   # (N, L, 1024) uint8
X_audio_q = np.load(SRC_DIR / 'X_audio.npy', mmap_mode='r')   # (N, L, 128)  uint8
lengths   = np.load(SRC_DIR / 'lengths.npy')                  # (N,)
y         = np.load(SRC_DIR / 'y.npy')                        # (N,)
N         = len(y)
print(f'N={N:,}  L={L}  N_CLASSES={N_CLASSES}')

N=4,939  L=60  N_CLASSES=12


In [2]:
# ============================================================
# 3.1 — Split (70/15/15, stratified)
# ============================================================
idx = np.arange(N)
idx_train, idx_tmp = train_test_split(idx, test_size=0.30,
                                       stratify=y, random_state=42)
idx_val, idx_test  = train_test_split(idx_tmp, test_size=0.50,
                                       stratify=y[idx_tmp], random_state=42)

print(f'Train: {len(idx_train):>6}  Val: {len(idx_val):>6}  Test: {len(idx_test):>6}')
print(f'\n  {"жанр":<14} {"train":>6} {"val":>5} {"test":>5}')
print('  ' + '─' * 34)
for i, g in enumerate(GENRES):
    tr = (y[idx_train]==i).sum(); va = (y[idx_val]==i).sum(); te = (y[idx_test]==i).sum()
    print(f'  {g:<14} {tr:>6} {va:>5} {te:>5}')

for name, arr in [('idx_train', idx_train), ('idx_val', idx_val), ('idx_test', idx_test)]:
    np.save(STAGE3_DIR / f'{name}.npy', arr)

Train:   3457  Val:    741  Test:    741

  жанр            train   val  test
  ──────────────────────────────────
  Animals           323    70    69
  Animation         324    69    69
  Beauty            108    23    23
  Dance             323    70    69
  Film              177    37    38
  Food              323    69    70
  Gaming            323    69    70
  Music             323    70    69
  Performance       323    69    70
  Sports            324    69    69
  Tech              263    56    56
  Vehicles          323    70    69


In [3]:
# ============================================================
# 3.2 — Normalization: mean/std per-feature на train
# ============================================================
# Считаем на *реальных* кадрах (без паддинга). Разворачиваем (N, L, D) → (N*L, D)
# с маской по lengths, затем mean/std по оси 0.

def compute_stats(X_q, lengths_arr, indices, dim_name):
    """Stream-считаем mean/std по всем реальным кадрам train."""
    D = X_q.shape[-1]
    n_total   = 0
    sum_      = np.zeros(D, dtype=np.float64)
    sum_sq    = np.zeros(D, dtype=np.float64)
    for i in indices:
        T = int(lengths_arr[i])
        if T == 0: continue
        x = np.asarray(X_q[i, :T], dtype=np.float32) * DEQ_S + DEQ_B   # (T, D)
        sum_   += x.sum(axis=0)
        sum_sq += (x * x).sum(axis=0)
        n_total += T
    mean = (sum_ / n_total).astype(np.float32)
    var  = (sum_sq / n_total - mean**2).clip(min=1e-6)
    std  = np.sqrt(var).astype(np.float32)
    print(f'  {dim_name:<6} stats: frames={n_total:,}  '
          f'mean∈[{mean.min():+.3f},{mean.max():+.3f}]  '
          f'std∈[{std.min():.3f},{std.max():.3f}]')
    return mean, std

rgb_mean, rgb_std = compute_stats(X_rgb_q,   lengths, idx_train, 'rgb')
aud_mean, aud_std = compute_stats(X_audio_q, lengths, idx_train, 'audio')

np.savez(STAGE3_DIR / 'norm_stats.npz',
         rgb_mean=rgb_mean, rgb_std=rgb_std,
         aud_mean=aud_mean, aud_std=aud_std)
print('Saved norm_stats.npz')

  rgb    stats: frames=207,302  mean∈[-0.114,+0.126]  std∈[0.894,0.978]
  audio  stats: frames=207,302  mean∈[-0.062,+0.215]  std∈[0.961,1.385]
Saved norm_stats.npz


In [4]:
# ============================================================
# 3.3 — Class weights
# ============================================================
counts_train = np.bincount(y[idx_train], minlength=N_CLASSES).astype(np.float64)
class_weights = 1.0 / np.sqrt(counts_train.clip(min=1))
class_weights = class_weights / class_weights.sum() * N_CLASSES
class_weights = class_weights.astype(np.float32)

print(f'  {"жанр":<14} {"train N":>8} {"weight":>8}')
for i, g in enumerate(GENRES):
    print(f'  {g:<14} {int(counts_train[i]):>8} {class_weights[i]:>8.4f}')

np.save(STAGE3_DIR / 'class_weights.npy', class_weights)
torch.save(torch.tensor(class_weights), STAGE3_DIR / 'class_weights.pt')

  жанр            train N   weight
  Animals             323   0.9101
  Animation           324   0.9087
  Beauty              108   1.5739
  Dance               323   0.9101
  Film                177   1.2294
  Food                323   0.9101
  Gaming              323   0.9101
  Music               323   0.9101
  Performance         323   0.9101
  Sports              324   0.9087
  Tech                263   1.0086
  Vehicles            323   0.9101


In [5]:
# ============================================================
# 3.4 — FrameDataset: dequantize + normalize на лету
# ============================================================
class FrameDataset(Dataset):
    """Отдаёт (rgb[L,D_r], audio[L,D_a], length, label).
    Паддированные кадры уже нули (после нормализации — ~mean/std из train, но
    мы оставляем их нулями чтобы RNN-маска/flatten могли их игнорировать)."""

    def __init__(self, rgb_q, aud_q, lengths, y, indices,
                 rgb_mean, rgb_std, aud_mean, aud_std,
                 deq_s, deq_b, augment=False, drop_audio_p=0.0):
        self.rgb_q    = rgb_q
        self.aud_q    = aud_q
        self.lengths  = lengths
        self.y        = y
        self.indices  = np.asarray(indices, dtype=np.int64)
        self.rgb_mean = rgb_mean.astype(np.float32)
        self.rgb_std  = rgb_std.astype(np.float32)
        self.aud_mean = aud_mean.astype(np.float32)
        self.aud_std  = aud_std.astype(np.float32)
        self.deq_s    = deq_s
        self.deq_b    = deq_b
        self.augment  = augment
        self.drop_audio_p = drop_audio_p

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        T   = int(self.lengths[idx])

        rgb = np.asarray(self.rgb_q[idx], dtype=np.float32) * self.deq_s + self.deq_b
        aud = np.asarray(self.aud_q[idx], dtype=np.float32) * self.deq_s + self.deq_b
        # нормализация на валидных кадрах; паддинг оставляем нулями
        rgb[:T] = (rgb[:T] - self.rgb_mean) / self.rgb_std
        aud[:T] = (aud[:T] - self.aud_mean) / self.aud_std
        rgb[T:] = 0.0
        aud[T:] = 0.0

        # имитация отсутствия аудио (как в заметках — подаём нули)
        if self.drop_audio_p > 0 and np.random.rand() < self.drop_audio_p:
            aud[:] = 0.0

        if self.augment:
            rgb[:T] += np.random.randn(*rgb[:T].shape).astype(np.float32) * 0.02
            aud[:T] += np.random.randn(*aud[:T].shape).astype(np.float32) * 0.03

        return (torch.from_numpy(rgb),
                torch.from_numpy(aud),
                torch.tensor(T, dtype=torch.long),
                torch.tensor(int(self.y[idx]), dtype=torch.long))


def make_datasets(augment_train=True):
    d = np.load(STAGE3_DIR / 'norm_stats.npz')
    kw = dict(rgb_mean=d['rgb_mean'], rgb_std=d['rgb_std'],
              aud_mean=d['aud_mean'], aud_std=d['aud_std'],
              deq_s=DEQ_S, deq_b=DEQ_B)
    train = FrameDataset(X_rgb_q, X_audio_q, lengths, y, idx_train,
                         augment=augment_train, drop_audio_p=0.1, **kw)
    val   = FrameDataset(X_rgb_q, X_audio_q, lengths, y, idx_val,   **kw)
    test  = FrameDataset(X_rgb_q, X_audio_q, lengths, y, idx_test,  **kw)
    return train, val, test

# sanity
ds_tr, ds_va, ds_te = make_datasets()
rgb, aud, T, yi = ds_tr[0]
print(f'sample[0]: rgb={tuple(rgb.shape)} aud={tuple(aud.shape)} T={T.item()} y={yi.item()}')
print(f'rgb mean/std on valid frames: {rgb[:T].mean():.3f} / {rgb[:T].std():.3f}')
print(f'rgb padding = 0: {bool((rgb[T:].abs().sum() == 0).item())}')

sample[0]: rgb=(60, 1024) aud=(60, 128) T=60 y=0
rgb mean/std on valid frames: -0.004 / 1.077
rgb padding = 0: True


In [6]:
# ============================================================
# 3.5 — Конфиг препроцессинга
# ============================================================
preproc_cfg = {
    'n_train'   : len(idx_train),
    'n_val'     : len(idx_val),
    'n_test'    : len(idx_test),
    'n_classes' : N_CLASSES,
    'L'         : L,
    'dim_rgb'   : DIM_RGB,
    'dim_audio' : DIM_AUD,
    'genres'    : GENRES,
    'class_weights': class_weights.tolist(),
}
with open(STAGE3_DIR / 'config.json', 'w') as f:
    json.dump(preproc_cfg, f, indent=2, ensure_ascii=False)
print('Stage 3 done. Artifacts:')
for p in sorted(STAGE3_DIR.iterdir()):
    print(f'  {p.name:<22} {p.stat().st_size/1024:.1f} KB')

Stage 3 done. Artifacts:
  class_weights.npy      0.2 KB
  class_weights.pt       1.6 KB
  config.json            0.6 KB
  idx_test.npy           5.9 KB
  idx_train.npy          27.1 KB
  idx_val.npy            5.9 KB
  norm_stats.npz         10.0 KB
